Unity Catalog setup for the Adventure Works warehouse.

Notebook contents:

* Create catalog `adwm_wh`
* Create core schemas for `bronze`, `silver`, `gold`, and `utilities`
* Create the `volumes` schema and landing/checkpoint volumes

Prerequisites:

* Run [external_location_setup](#notebook-465211359346104) successfully first
* External location and storage permissions must already be configured in Azure
* Managed location and volume paths must point to valid storage containers

Expected result:

* A ready-to-use Unity Catalog structure for the ingestion and warehouse notebooks

In [0]:
# Create input widgets for storage account and container
dbutils.widgets.text('storage_account', 'adwgen2')
dbutils.widgets.text('container', 'uc-adwm')

# Use widget values in variables
storage_account = dbutils.widgets.get('storage_account')
container = dbutils.widgets.get('container')

In [0]:
catalog_sql = f"""
CREATE CATALOG IF NOT EXISTS `adwm_wh`
MANAGED LOCATION 'abfss://{container}@{storage_account}.dfs.core.windows.net/adwm_wh'
COMMENT 'ETL workspace catalog for Adventure Works data'
"""

spark.sql(catalog_sql)

In [0]:
%sql
    
-- Create schemas in the new catalog
CREATE SCHEMA IF NOT EXISTS adwm_wh.bronze COMMENT 'Raw/landing zone data';
CREATE SCHEMA IF NOT EXISTS adwm_wh.silver COMMENT 'Cleansed and conformed data';
CREATE SCHEMA IF NOT EXISTS adwm_wh.gold COMMENT 'Business-level aggregates';
CREATE SCHEMA IF NOT EXISTS adwm_wh.utilities COMMENT 'Helper tables and functions';

In [0]:
# Create volumes schema
spark.sql("CREATE SCHEMA IF NOT EXISTS adwm_wh.volumes COMMENT 'Schema for Unity Catalog volumes'")

# Create an EXTERNAL volume in the volumes schema using the correct LOCATION syntax
external_volume_sql = f"""
CREATE EXTERNAL VOLUME IF NOT EXISTS adwm_wh.volumes.landing_files
LOCATION 'abfss://sharif@{storage_account}.dfs.core.windows.net/'
COMMENT 'Volume for landing files in separate container'
"""

spark.sql(external_volume_sql)

# Create volume for storing streaming checkpoints
spark.sql("CREATE VOLUME IF NOT EXISTS adwm_wh.volumes.checkpoints COMMENT 'Storage for Auto Loader streaming checkpoints'")

Verification guidance:

* After running this notebook, confirm that catalog `adwm_wh` exists
* Confirm schemas `bronze`, `silver`, `gold`, `utilities`, and `volumes` are present
* Confirm the landing and checkpoint volumes are available under `adwm_wh.volumes`

These checks can be performed interactively in Catalog Explorer or with separate ad hoc SQL when needed.